# Model comparison for with vs without radiation_day_before

In [ ]:
# imports
import numpy as np
import pandas as pd
from pathlib import Path
from sklearn.metrics import mean_squared_error, mean_absolute_error, mean_absolute_percentage_error, r2_score

DATA_DIR = Path("../data/NSW")
RESULTS_DIR = Path("../results")

In [ ]:
# reading in 2019 data so we have the associated features, etc
actual = pd.read_csv(DATA_DIR / "nsw_features_added.csv", parse_dates=["DATETIME"])
actual = actual[actual["DATETIME"].dt.year == 2019][["DATETIME", "TOTALDEMAND", "TEMPERATURE"]]

actual.shape

(17520, 3)

In [ ]:
# just including the eseasons
season_by_month = {
    12: "Summer", 1: "Summer", 2: "Summer",
    3: "Autumn", 4: "Autumn", 5: "Autumn",
    6: "Winter", 7: "Winter", 8: "Winter",
    9: "Spring", 10: "Spring", 11: "Spring",
}
actual["season"] = actual["DATETIME"].dt.month.map(season_by_month)
actual["temp_band"] = np.where(actual["TEMPERATURE"] >= 18, "above_18", "below_18")

actual.head()

,DATETIME,TOTALDEMAND,TEMPERATURE,season,temp_band
157776,2019-01-01 00:00:00,7612.74,22.3,Summer,above_18
157777,2019-01-01 00:30:00,7457.58,22.3,Summer,above_18
157778,2019-01-01 01:00:00,7243.21,23.0,Summer,above_18
157779,2019-01-01 01:30:00,6918.55,23.2,Summer,above_18
157780,2019-01-01 02:00:00,6676.58,23.8,Summer,above_18


In [ ]:
# just poicking which csvs to look at 

MODEL_STEMS = ["baseline", "lightgbm", "xgboost", "prophet", "random_forest"]

VARIANTS = {
    "with": "{stem}_predictions.csv",
    "without": "{stem}_no_radiation_day_before_predictions.csv",
}

In [ ]:
# some helpful functions just so it can calcualte our stats quicky

def evaluate(y_true, y_pred):
    rmse = np.sqrt(mean_squared_error(y_true, y_pred))
    mae = mean_absolute_error(y_true, y_pred)
    mape = mean_absolute_percentage_error(y_true, y_pred) * 100
    r2 = r2_score(y_true, y_pred)
    return {"rmse": rmse, "mae": mae, "mape_pct": mape, "r2": r2}


def segment_stats(model_name, with_radiation, merged):
    segments = {"overall": merged}
    for season in ["Summer", "Autumn", "Winter", "Spring"]:
        segments[season] = merged[merged["season"] == season]
    segments["below_18"] = merged[merged["temp_band"] == "below_18"]
    segments["above_18"] = merged[merged["temp_band"] == "above_18"]

    rows = []
    for segment_name, seg_df in segments.items():
        stats = evaluate(seg_df["TOTALDEMAND"], seg_df["prediction"])
        rows.append({"model_name": model_name, "with_radiation": with_radiation, "segment": segment_name, **stats})
    return rows

In [ ]:
# outputs
rows = []

for stem in MODEL_STEMS:
    for variant, pattern in VARIANTS.items():
        path = RESULTS_DIR / pattern.format(stem=stem)
        if not path.exists(): # i have added this here efor now as we dont have day_before for all
            continue

        preds = pd.read_csv(path)
        preds["DATETIME"] = pd.to_datetime(preds["DATETIME"], format="mixed", dayfirst=True)
        pred_col = [c for c in preds.columns if c != "DATETIME"][0]
        preds = preds.rename(columns={pred_col: "prediction"})

        merged = actual.merge(preds, on="DATETIME", how="inner")
        rows.extend(segment_stats(stem, variant == "with", merged))

comparison = pd.DataFrame(rows)
comparison.shape

skipping baseline (without) - baseline_no_radiation_day_before_predictions.csv not found yet
skipping lightgbm (without) - lightgbm_no_radiation_day_before_predictions.csv not found yet
skipping xgboost (without) - xgboost_no_radiation_day_before_predictions.csv not found yet
skipping prophet (without) - prophet_no_radiation_day_before_predictions.csv not found yet


(42, 7)

In [ ]:
# save
comparison.to_csv(RESULTS_DIR / "model_comparison_by_segment.csv", index=False)
comparison

,model_name,with_radiation,segment,rmse,mae,mape_pct,r2
0,baseline,True,overall,535.989120,380.215288,4.743792,0.816091
1,baseline,True,Summer,752.240808,523.904104,6.098386,0.756580
2,baseline,True,Autumn,415.977387,307.099323,4.036791,0.821120
3,baseline,True,Winter,458.510586,357.565800,4.282996,0.844860
4,baseline,True,Spring,452.951988,334.923290,4.584713,0.732202
5,baseline,True,below_18,411.590405,307.569435,3.926999,0.893283
6,baseline,True,above_18,630.381398,448.065665,5.506666,0.741206
7,lightgbm,True,overall,463.995139,307.403004,3.744587,0.862178
8,lightgbm,True,Summer,672.850961,452.181142,5.151782,0.805249
9,lightgbm,True,Autumn,328.190565,218.679987,2.792502,0.888654
